# Xử lý ngôn ngữ tự nhiên - CS221.Q21.KHTN

## Demo chương 6: Neural Networks

### Nhóm 1:
- Bảo Quý Định Tân - 24520028
- Lê Văn Thức - 24521748
- Lê Phạm Thành Nhân - 24520022

### Ở demo cho chương Neural Networks này ta sẽ sử dụng lại bài toán ở demo của chương 4 trên dữ liệu SA-Restaurant

# Khai báo thư viện

In [1]:
import re
import numpy as np
import cupy as cp
import cupyx.scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.dummy import DummyClassifier
import warnings

warnings.filterwarnings('ignore')

# Chuẩn bị data

In [2]:
# ==========================================
# Data Loading & Preprocessing (CPU)
# ==========================================
def load_data(filepath):
    texts = []
    labels = []
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')
        
    for i in range(0, len(lines), 4):
        if i + 2 >= len(lines):
            break
        text = lines[i+1].strip()
        label_line = lines[i+2].strip()
        
        aspect_dict = {}
        if label_line:
            matches = re.findall(r'\{([^,]+),\s*([^}]+)\}', label_line)
            for aspect, polarity in matches:
                aspect_dict[aspect.strip()] = polarity.strip()
                
        texts.append(text)
        labels.append(aspect_dict)
        
    return texts, labels

# Load data
train_texts, train_labels = load_data('VLSP2018-SA-train-dev-test/1-VLSP2018-SA-Restaurant-train (7-3-2018).txt')
dev_texts, dev_labels = load_data('VLSP2018-SA-train-dev-test/2-VLSP2018-SA-Restaurant-dev (7-3-2018).txt')
test_texts, test_labels = load_data('VLSP2018-SA-train-dev-test/3-VLSP2018-SA-Restaurant-test (8-3-2018).txt')

entities = [
    "RESTAURANT", 
    "AMBIENCE", 
    "LOCATION", 
    "FOOD", 
    "SERVICE", 
    "DRINKS"
]

attributes = [
    "GENERAL", 
    "PRICES", 
    "QUALITY", 
    "STYLE&OPTIONS", 
    "MISCELLANEOUS"
]
all_aspects = sorted([f"{e}#{a}" for e in entities for a in attributes])

def extract_aspect_labels(labels_list, aspects):
    y_dict = {aspect: [] for aspect in aspects}
    for label_dict in labels_list:
        for aspect in aspects:
            y_dict[aspect].append(label_dict.get(aspect, 'null'))
    return y_dict

y_train_dict = extract_aspect_labels(train_labels, all_aspects)
y_dev_dict = extract_aspect_labels(dev_labels, all_aspects)
y_test_dict = extract_aspect_labels(test_labels, all_aspects)

# TF-IDF Vectorization
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000)
X_train_cpu = vectorizer.fit_transform(train_texts)
X_dev_cpu = vectorizer.transform(dev_texts)
X_test_cpu = vectorizer.transform(test_texts)

# ==========================================
# Dense Arrays + Float32
# ==========================================
print("Converting to Dense Float32 Arrays and moving to GPU...")
X_train_gpu = cp.array(X_train_cpu.toarray(), dtype=cp.float32)
X_dev_gpu = cp.array(X_dev_cpu.toarray(), dtype=cp.float32)
X_test_gpu = cp.array(X_test_cpu.toarray(), dtype=cp.float32)


Converting to Dense Float32 Arrays and moving to GPU...


# Tạo mô hình


### Nhưng mô hình ta sử dụng sẽ là một FFNN (Feed forward Neural Network) đơn giản

Bao gồm một lớp ẩn (hidden layer) sử dụng hàm kích hoạt ReLU và một lớp đầu ra (output layer) sử dụng hàm kích hoạt Sigmoid.

$$
\begin{align*}
\mathbf{z}^{[1]} &= \mathbf{W}^{[1]}\mathbf{x} + \mathbf{b}^{[1]} \\
\mathbf{a}^{[1]} &= \text{ReLU}(\mathbf{z}^{[1]}) \\
z^{[2]} &= \mathbf{W}^{[2]}\mathbf{a}^{[1]} + b^{[2]} \\
a^{[2]} &= \sigma(z^{[2]}) \\
\hat{y} &= a^{[2]}
\end{align*}
$$

### Các bước đạo hàm lan truyền ngược (Backpropagation)

Để huấn luyện mô hình sử dụng các thuật toán liên quan đến Gradient Descent thì ta cần phải lấy được đạo hàm trên các tham số (parameters).

Giả sử sử dụng hàm mất mát Binary Cross-Entropy $L = -[y \log(a^{[2]}) + (1-y)\log(1-a^{[2]})]$.

Gọi $dZ$ là ký hiệu rút gọn cho đạo hàm riêng của hàm mất mát theo $Z$ ($\frac{\partial L}{\partial Z}$).

**1. Lớp đầu ra (Layer 2)**

* **Đạo hàm theo giá trị kích hoạt $a^{[2]}$:**

    $$da^{[2]} = \frac{\partial L}{\partial a^{[2]}} = -\frac{y}{a^{[2]}} + \frac{1-y}{1-a^{[2]}} = \frac{a^{[2]} - y}{a^{[2]}(1-a^{[2]})}$$

* **Đạo hàm theo $z^{[2]}$:**

    Vì $a^{[2]} = \sigma(z^{[2]})$, đạo hàm của hàm sigmoid là $\sigma'(z^{[2]}) = a^{[2]}(1-a^{[2]})$.
    $$dz^{[2]} = da^{[2]} \cdot \sigma'(z^{[2]}) = \left( \frac{a^{[2]} - y}{a^{[2]}(1-a^{[2]})} \right) \cdot a^{[2]}(1-a^{[2]}) = a^{[2]} - y$$

* **Đạo hàm theo $W$ và $b$ lớp 2:**

    Từ phương trình $z^{[2]} = W^{[2]}a^{[1]} + b^{[2]}$, áp dụng chain rule với giá trị đầu vào chuyển vị:
    $$dW^{[2]} = dz^{[2]} a^{[1]T}$$
    $$db^{[2]} = dz^{[2]}$$

**2. Lớp ẩn (Layer 1)**

* **Đạo hàm theo $a^{[1]}$:**

    Ta truyền phần lỗi (error) ngược trở lại qua ma trận trọng số của lớp 2.
    $$da^{[1]} = W^{[2]T} dz^{[2]}$$

* **Đạo hàm theo $z^{[1]}$:**

    Vì $a^{[1]} = \text{ReLU}(z^{[1]})$, ta nhân từng phần tử (element-wise, ký hiệu $\ast$) với đạo hàm của hàm ReLU.
    $$dz^{[1]} = da^{[1]} \ast \text{ReLU}'(z^{[1]})$$

* **Đạo hàm theo $W$ và $b$ lớp 1:**

    Từ phương trình $z^{[1]} = W^{[1]}x + b^{[1]}$, áp dụng chain rule:
    $$dW^{[1]} = dz^{[1]} x^T$$
    $$db^{[1]} = dz^{[1]}$$

In [3]:
class CuPyFeedForwardNeuralNetwork:
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.001, epochs=100, batch_size=32):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        
        # Adam Optimizer hyperparameters
        self.beta1 = 0.9
        self.beta2 = 0.999
        self.epsilon = 1e-8
        self.t = 0  # Timestep for Adam
        
        self.W1, self.b1, self.W2, self.b2 = self._initialize_parameters(input_size, hidden_size, output_size)
        self._initialize_adam_moments(input_size, hidden_size, output_size)

    def _initialize_parameters(self, input_size, hidden_size, output_size):
        cp.random.seed(42)
        # He Initialization for ReLU (scaled by sqrt(2/input_size))
        W1 = cp.random.randn(input_size, hidden_size, dtype=cp.float32) * cp.sqrt(2.0 / input_size, dtype=cp.float32)
        b1 = cp.zeros((1, hidden_size), dtype=cp.float32)
        # He Initialization for the output layer
        W2 = cp.random.randn(hidden_size, output_size, dtype=cp.float32) * cp.sqrt(2.0 / hidden_size, dtype=cp.float32)
        b2 = cp.zeros((1, output_size), dtype=cp.float32)
        return W1, b1, W2, b2

    def _initialize_adam_moments(self, input_size, hidden_size, output_size):
        # First moment (momentum)
        self.v_dW1 = cp.zeros((input_size, hidden_size), dtype=cp.float32)
        self.v_db1 = cp.zeros((1, hidden_size), dtype=cp.float32)
        self.v_dW2 = cp.zeros((hidden_size, output_size), dtype=cp.float32)
        self.v_db2 = cp.zeros((1, output_size), dtype=cp.float32)
        
        # Second moment (RMSprop)
        self.s_dW1 = cp.zeros((input_size, hidden_size), dtype=cp.float32)
        self.s_db1 = cp.zeros((1, hidden_size), dtype=cp.float32)
        self.s_dW2 = cp.zeros((hidden_size, output_size), dtype=cp.float32)
        self.s_db2 = cp.zeros((1, output_size), dtype=cp.float32)

    def _relu(self, z):
        return cp.maximum(0, z)

    def _relu_derivative(self, z):
        return cp.where(z > 0, cp.float32(1), cp.float32(0))

    def _softmax(self, z):
        exp_z = cp.exp(z - cp.max(z, axis=1, keepdims=True)) 
        return exp_z / cp.sum(exp_z, axis=1, keepdims=True)

    def _forward_propagation(self, x):
        self.z1 = x @ self.W1 + self.b1
        self.a1 = self._relu(self.z1)
        
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = self._softmax(self.z2)
        return self.a2

    def _backward_propagation(self, x, y_true_onehot):
        m = y_true_onehot.shape[0]
        
        dz2 = self.a2 - y_true_onehot
        dW2 = (self.a1.T @ dz2) / m 
        db2 = cp.sum(dz2, axis=0, keepdims=True) / m
        
        da1 = dz2 @ self.W2.T
        dz1 = da1 * self._relu_derivative(self.z1)
        
        dW1 = (x.T @ dz1) / m 
        db1 = cp.sum(dz1, axis=0, keepdims=True) / m
        
        return dW1, db1, dW2, db2

    def _update_parameters(self, dW1, db1, dW2, db2):
        self.t += 1
        
        # Update Adam moments for W1
        self.v_dW1 = self.beta1 * self.v_dW1 + (1 - self.beta1) * dW1
        self.s_dW1 = self.beta2 * self.s_dW1 + (1 - self.beta2) * (dW1 ** 2)
        v_dW1_corr = self.v_dW1 / (1 - self.beta1 ** self.t)
        s_dW1_corr = self.s_dW1 / (1 - self.beta2 ** self.t)
        self.W1 -= self.learning_rate * v_dW1_corr / (cp.sqrt(s_dW1_corr) + self.epsilon)

        # Update Adam moments for b1
        self.v_db1 = self.beta1 * self.v_db1 + (1 - self.beta1) * db1
        self.s_db1 = self.beta2 * self.s_db1 + (1 - self.beta2) * (db1 ** 2)
        v_db1_corr = self.v_db1 / (1 - self.beta1 ** self.t)
        s_db1_corr = self.s_db1 / (1 - self.beta2 ** self.t)
        self.b1 -= self.learning_rate * v_db1_corr / (cp.sqrt(s_db1_corr) + self.epsilon)

        # Update Adam moments for W2
        self.v_dW2 = self.beta1 * self.v_dW2 + (1 - self.beta1) * dW2
        self.s_dW2 = self.beta2 * self.s_dW2 + (1 - self.beta2) * (dW2 ** 2)
        v_dW2_corr = self.v_dW2 / (1 - self.beta1 ** self.t)
        s_dW2_corr = self.s_dW2 / (1 - self.beta2 ** self.t)
        self.W2 -= self.learning_rate * v_dW2_corr / (cp.sqrt(s_dW2_corr) + self.epsilon)

        # Update Adam moments for b2
        self.v_db2 = self.beta1 * self.v_db2 + (1 - self.beta1) * db2
        self.s_db2 = self.beta2 * self.s_db2 + (1 - self.beta2) * (db2 ** 2)
        v_db2_corr = self.v_db2 / (1 - self.beta1 ** self.t)
        s_db2_corr = self.s_db2 / (1 - self.beta2 ** self.t)
        self.b2 -= self.learning_rate * v_db2_corr / (cp.sqrt(s_db2_corr) + self.epsilon)

    def fit(self, x_train_gpu, y_train_labels):
        self.classes_ = sorted(list(set(y_train_labels)))
        n_classes = len(self.classes_)
        
        y_to_idx = {label: i for i, label in enumerate(self.classes_)}
        y_train_idx = cp.array([y_to_idx[y] for y in y_train_labels], dtype=cp.int32)
        y_train_onehot = cp.eye(n_classes, dtype=cp.float32)[y_train_idx]
        
        m = x_train_gpu.shape[0]
        self.t = 0 # Reset timestep for new fitting
        
        for epoch in range(self.epochs):
            for i in range(0, m, self.batch_size):
                batch_x = x_train_gpu[i:i+self.batch_size]
                batch_y_onehot = y_train_onehot[i:i+self.batch_size]
                
                if batch_x.shape[0] == 0:
                    continue
                    
                _ = self._forward_propagation(batch_x)
                dW1, db1, dW2, db2 = self._backward_propagation(batch_x, batch_y_onehot)
                self._update_parameters(dW1, db1, dW2, db2)

    def evaluate_accuracy_gpu(self, x_gpu, y_true_labels):
        probs = self._forward_propagation(x_gpu)
        pred_indices = cp.argmax(probs, axis=1)
        
        y_to_idx = {label: i for i, label in enumerate(self.classes_)}
        y_true_idx = cp.array([y_to_idx.get(y, -1) for y in y_true_labels], dtype=cp.int32)
        
        accuracy = cp.mean(pred_indices == y_true_idx).item()
        return accuracy

    def predict(self, x_gpu):
        probs = self._forward_propagation(x_gpu)
        pred_indices = cp.argmax(probs, axis=1)
        pred_indices_cpu = pred_indices.get() 
        return np.array([self.classes_[i] for i in pred_indices_cpu])

### Huấn luyện mô hình

In [8]:
# ==========================================
# GPU Grid Search & Training Loop
# ==========================================
models = {}
best_params = {}

# Notice the learning rates are smaller here (0.001 is standard for Adam)
hyperparam_grid = [
    {'learning_rate': 0.001, 'epochs': 50, 'hidden_size': 64, 'batch_size': 32},
    {'learning_rate': 0.001, 'epochs': 100, 'hidden_size': 64, 'batch_size': 32},
    {'learning_rate': 0.0005, 'epochs': 50, 'hidden_size': 64, 'batch_size': 32},
]

print("Đang tìm kiếm siêu tham số tốt nhất trên GPU...")
input_size = X_train_gpu.shape[1]

for aspect in all_aspects:
    y_train_aspect = y_train_dict[aspect]
    y_dev_aspect = y_dev_dict[aspect]
    
    unique_train_classes = set(y_train_aspect)
    
    if len(unique_train_classes) == 1:
        dummy_model = DummyClassifier(strategy='constant', constant=list(unique_train_classes)[0])
        dummy_model.fit(X_train_cpu, y_train_aspect)
        models[aspect] = dummy_model
        best_params[aspect] = "N/A (Dummy)"
        continue
    
    n_classes = len(unique_train_classes)
    best_hp = hyperparam_grid[0]
    best_acc = -1.0
    
    for params in hyperparam_grid:
        temp_model = CuPyFeedForwardNeuralNetwork(
            input_size=input_size, 
            hidden_size=params['hidden_size'],
            output_size=n_classes,
            learning_rate=params['learning_rate'],
            epochs=params['epochs'],
            batch_size=params['batch_size']
        )
        temp_model.fit(X_train_gpu, y_train_aspect)
        
        acc = temp_model.evaluate_accuracy_gpu(X_dev_gpu, y_dev_aspect)
        
        if acc > best_acc:
            best_acc = acc
            best_hp = params
            
    best_params[aspect] = best_hp
    
    final_model = CuPyFeedForwardNeuralNetwork(
        input_size=input_size, 
        hidden_size=best_hp['hidden_size'],
        output_size=n_classes,
        learning_rate=best_hp['learning_rate'],
        epochs=best_hp['epochs'],
        batch_size=best_hp['batch_size']
    )
    final_model.fit(X_train_gpu, y_train_aspect)
    models[aspect] = final_model
    
print("Hoàn tất việc tối ưu và huấn luyện trên GPU!\n")

# ==========================================
# 4. Final Evaluation on Test Set
# ==========================================
def evaluate_models(X_gpu, X_cpu, y_dict_true, aspects, models):
    print(f"{'='*60}")
    print(f"{'ASPECT':<30} | {'ACCURACY':<10} | {'MACRO F1':<10}")
    print(f"{'-'*60}")
    
    acc_list, f1_list = [], []
    
    for aspect in aspects:
        model = models[aspect]
        y_true = y_dict_true[aspect]
        
        if isinstance(model, DummyClassifier):
            y_pred = model.predict(X_cpu)
        else:
            y_pred = model.predict(X_gpu)
            
        unique_labels = set(y_true)
        if len(unique_labels) > 1 or 'null' not in unique_labels:
            acc = accuracy_score(y_true, y_pred)
            f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
            
            acc_list.append(acc)
            f1_list.append(f1)
            print(f"{aspect:<30} | {acc:.4f}     | {f1:.4f}")
            
    print(f"{'='*60}")
    if acc_list:
        print(f"{'TRUNG BÌNH TỔNG THỂ':<30} | {sum(acc_list)/len(acc_list):.4f}     | {sum(f1_list)/len(f1_list):.4f}")

evaluate_models(X_test_gpu, X_test_cpu, y_test_dict, all_aspects, models)

Đang tìm kiếm siêu tham số tốt nhất trên GPU...
Hoàn tất việc tối ưu và huấn luyện trên GPU!

ASPECT                         | ACCURACY   | MACRO F1  
------------------------------------------------------------
AMBIENCE#GENERAL               | 0.6540     | 0.3393
DRINKS#PRICES                  | 0.8480     | 0.2294
DRINKS#QUALITY                 | 0.8620     | 0.2492
DRINKS#STYLE&OPTIONS           | 0.9080     | 0.2379
FOOD#PRICES                    | 0.4540     | 0.3524
FOOD#QUALITY                   | 0.8340     | 0.3750
FOOD#STYLE&OPTIONS             | 0.6860     | 0.3135
LOCATION#GENERAL               | 0.6440     | 0.2054
RESTAURANT#GENERAL             | 0.6400     | 0.3165
RESTAURANT#MISCELLANEOUS       | 0.7400     | 0.2126
RESTAURANT#PRICES              | 0.8540     | 0.2303
SERVICE#GENERAL                | 0.7420     | 0.3555
TRUNG BÌNH TỔNG THỂ            | 0.7388     | 0.2848


In [7]:
# ==========================================
# Test with Custom Sentences
# ==========================================
def predict_custom_text(text, vectorizer, models, aspects):
    # 1. Vectorize the text on CPU
    X_custom_cpu = vectorizer.transform([text])
    
    # 2. Move to GPU
    X_custom_gpu = cp.array(X_custom_cpu.toarray(), dtype=cp.float32)
    
    print(f"\nAnalyzing: '{text}'")
    print("-" * 40)
    
    found_aspects = False
    for aspect in aspects:
        model = models[aspect]
        
        if isinstance(model, DummyClassifier):
            pred = model.predict(X_custom_cpu)[0]
        else:
            pred = model.predict(X_custom_gpu)[0]
            
        # Only print if the model detects an actual sentiment (ignores 'null')
        if pred != 'null':
            print(f"-> {aspect}: {pred}")
            found_aspects = True
            
    if not found_aspects:
        print("No specific aspects detected.")

# Example Usage:
sample_text = "Đây là 1 trong những quán mà mình thích vì vị trà đậm và thơm cũng như mùi vị đặc trưng hơn hẳn những quán khác nè  Trà sữa trân châu sợi - 46k Trà sữa pha khá ngon, vị trà chát và mùi hương khá rõ, không quá ngọt, rất đúng với gu mình  Trà đào - 45k Vị trà đào ở đây cũng đặc biệt hơn hẳn những quán khác, không phải chua ngọt như thưởng thấy mà có mùi trà rất ngon  Cà phê đá xay - 65k Món đá xay ở đây uống cũng ngon không kém trà nè, mùi vị thơm hương cà phê, vị đắng kết hợp hoàn hảo với độ béo ngọt của whipping cream, không quá đắng, cũng không quá ngọt hay lạt lẽo mà dịu nhẹ, thơm và dễ uống lắm  Trà vải thiết quan âm - 45k Trà vải có mùi vị rất thơm ngon mùi vải mà vẫn nghe rõ vị trà, có chút vị chát nhẹ mùi trà thơm rất thích, không phải chỉ toàn vị syrup vải ngọt gắt như nhiều chỗ khác. Do trà ở đây pha khá đậm nên bạn nào uống mà đang đói sẽ dễ say nha, hoặc ban đêm có thể khó ngủ à, cảnh báo trước  Trà thiết quan âm latte - 47k Ly này thì vị trà rất đậm nên cảm giác hơi nhạt và chát, phần kem sữa bên trên béo béo uống chung thì vừa  Trà sữa 101 - 52k Ly này chụp ké chứ không có thử :v"
# "Chúng tôi không cảm thấy thoải mái vì ở chỉ 1 ngày mà cúp điện 3,4 lần . Thang máy thì quá nóng và quá chậm chạp. Bữa sáng tuyệt. Phòng ngủ tiện nghi đẹp. Nhân viên phục phụ rất tốt." # "Phòng rất sạch sẽ và giường ngủ thoải mái, nhưng giá hơi đắt." 
predict_custom_text(sample_text, vectorizer, models, all_aspects)


Analyzing: 'Đây là 1 trong những quán mà mình thích vì vị trà đậm và thơm cũng như mùi vị đặc trưng hơn hẳn những quán khác nè  Trà sữa trân châu sợi - 46k Trà sữa pha khá ngon, vị trà chát và mùi hương khá rõ, không quá ngọt, rất đúng với gu mình  Trà đào - 45k Vị trà đào ở đây cũng đặc biệt hơn hẳn những quán khác, không phải chua ngọt như thưởng thấy mà có mùi trà rất ngon  Cà phê đá xay - 65k Món đá xay ở đây uống cũng ngon không kém trà nè, mùi vị thơm hương cà phê, vị đắng kết hợp hoàn hảo với độ béo ngọt của whipping cream, không quá đắng, cũng không quá ngọt hay lạt lẽo mà dịu nhẹ, thơm và dễ uống lắm  Trà vải thiết quan âm - 45k Trà vải có mùi vị rất thơm ngon mùi vải mà vẫn nghe rõ vị trà, có chút vị chát nhẹ mùi trà thơm rất thích, không phải chỉ toàn vị syrup vải ngọt gắt như nhiều chỗ khác. Do trà ở đây pha khá đậm nên bạn nào uống mà đang đói sẽ dễ say nha, hoặc ban đêm có thể khó ngủ à, cảnh báo trước  Trà thiết quan âm latte - 47k Ly này thì vị trà rất đậm nên cảm giác